<a href="https://colab.research.google.com/github/bihagkashikar/bits-pilani-mtech-genai-ml/blob/master/maths-assignment-01/q2_power_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q2: Rank, Covariance and the Power Method
This notebook contains the complete self-contained Python solution for Q2.

## Q2.1: Generate Dataset


In [43]:
"""Q2.1: Generate X in R^(500 x 6) from four standard-normal features."""

import numpy as np


def generate_dataset(seed=41602):
    """Generate the four random features and the two dependent features."""
    rng = np.random.default_rng(seed)
    first_four = rng.standard_normal((500, 4))
    f1 = first_four[:, 0]
    f2 = first_four[:, 1]
    f3 = first_four[:, 2]
    f4 = first_four[:, 3]
    f5 = 2.0 * f1 + 3.0 * f2
    f6 = f3 - 2.0 * f4
    return np.column_stack((f1, f2, f3, f4, f5, f6))


def print_section(title):
    """Print a consistent section heading for the submitted output."""
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def print_vector(title, vector):
    """Print a labeled vector with eight decimal places."""
    print(title)
    print("[" + ", ".join(f"{value:.8f}" for value in vector) + "]")


def print_matrix(title, matrix):
    """Print a labeled matrix with one formatted row per line."""
    print(title)
    for row in matrix:
        print("[" + ", ".join(f"{value:.8f}" for value in row) + "]")


data = generate_dataset()
print_section("Q2.1 - DATASET GENERATION")
print("Random seed: 41602")
print("Dataset shape: X is 500 x 6")
print_matrix("First five rows of X:", data[:5])
print(
    "Maximum error in f5 = 2f1 + 3f2:",
    f"{np.max(np.abs(data[:, 4] - 2.0 * data[:, 0] - 3.0 * data[:, 1])):.8e}",
)
print(
    "Maximum error in f6 = f3 - 2f4:",
    f"{np.max(np.abs(data[:, 5] - data[:, 2] + 2.0 * data[:, 3])):.8e}",
)


Q2.1 - DATASET GENERATION
Random seed: 41602
Dataset shape: X is 500 x 6
First five rows of X:
[-0.04444922, -0.97737317, -0.86730583, 0.01608347, -3.02101796, -0.89947277]
[0.33201692, -1.45761124, 1.02776546, -0.51582919, -3.70879988, 2.05942383]
[-1.59190489, 0.61410815, 0.46316909, 0.37439560, -1.34148532, -0.28562212]
[0.03763868, -0.33246541, -0.50019994, -1.20679241, -0.92211887, 1.91338487]
[0.42596594, 0.19393104, 0.42128363, -0.19738786, 1.43372500, 0.81605935]
Maximum error in f5 = 2f1 + 3f2: 8.88178420e-16
Maximum error in f6 = f3 - 2f4: 8.88178420e-16


The generated dataset has 500 observations and 6 features. The first four features are standard-normal random values, while the last two are deterministic linear combinations. The maximum relation errors are at floating-point round-off level, confirming the definitions of `f5` and `f6`.

## Q2.2: Compute Rank


In [44]:
"""Q2.2: Compute the rank of X using Gaussian elimination."""


def matrix_rank_by_elimination(matrix, tolerance=1e-10):
    """Compute matrix rank without NumPy's matrix-rank helper."""
    work = matrix.astype(float).copy()
    row_count, column_count = work.shape
    pivot_row = 0
    rank = 0
    for column in range(column_count):
        if pivot_row == row_count:
            break
        candidate = pivot_row
        for row in range(pivot_row + 1, row_count):
            if abs(work[row, column]) > abs(work[candidate, column]):
                candidate = row
        if abs(work[candidate, column]) <= tolerance:
            continue
        work[[pivot_row, candidate]] = work[[candidate, pivot_row]]
        for lower_row in range(pivot_row + 1, row_count):
            multiplier = work[lower_row, column] / work[pivot_row, column]
            work[lower_row, column:] -= multiplier * work[pivot_row, column:]
        rank += 1
        pivot_row += 1
    return rank


rank_of_x = matrix_rank_by_elimination(data)
print_section("Q2.2 - RANK OF X")
print(f"Rank(X): {rank_of_x}")
print("Interpretation: two dependent columns leave four independent directions.")


Q2.2 - RANK OF X
Rank(X): 4
Interpretation: two dependent columns leave four independent directions.


Gaussian elimination reports rank 4. This is expected because `f5` and `f6` are linear combinations of the first four features, so the six columns contain only four independent directions.

## Q2.3(a): Covariance Matrix


In [45]:
"""Q2.3(a): Compute C = (1/n) X^T X."""


def covariance_matrix(data):
    """Compute the covariance matrix specified in the question."""
    return (data.T @ data) / data.shape[0]


covariance = covariance_matrix(data)
print_section("Q2.3(a) - COVARIANCE MATRIX")
print("Formula: C = (1 / n) X^T X, where n = 500")
print_matrix("Covariance matrix C (6 x 6):", covariance)


Q2.3(a) - COVARIANCE MATRIX
Formula: C = (1 / n) X^T X, where n = 500
Covariance matrix C (6 x 6):
[0.98945778, 0.07758642, 0.07723531, -0.06362116, 2.21167480, 0.20447763]
[0.07758642, 0.90408263, -0.08107655, -0.03884676, 2.86742072, -0.00338304]
[0.07723531, -0.08107655, 0.99906751, -0.04451261, -0.08875903, 1.08809274]
[-0.06362116, -0.03884676, -0.04451261, 0.94490821, -0.24378259, -1.93432904]
[2.21167480, 2.86742072, -0.08875903, -0.24378259, 13.02561177, 0.39880615]
[0.20447763, -0.00338304, 1.08809274, -1.93432904, 0.39880615, 4.95675082]


The covariance matrix is computed exactly as $C = \frac{1}{n}X^T X$ with $n = 500$. It is a symmetric $6 \times 6$ matrix. Because the dataset has rank 4, its covariance matrix has four nonzero eigenvalues and two zero eigenvalues, up to floating-point error.

## Q2.3(b-c): Power Method and Deflation


In [46]:
"""Q2.3(b-c): Approximate successive eigenpairs using power-method deflation."""


def power_method(matrix, tolerance=1e-7, max_iterations=100000):
    """Approximate the dominant eigenpair and return its iteration count."""
    vector = np.ones(matrix.shape[0], dtype=float)
    vector /= np.linalg.norm(vector)
    previous_eigenvalue = 0.0
    for iteration in range(1, max_iterations + 1):
        next_vector = matrix @ vector
        next_vector /= np.linalg.norm(next_vector)
        eigenvalue = float(next_vector @ matrix @ next_vector)
        if abs(eigenvalue - previous_eigenvalue) < tolerance:
            return eigenvalue, next_vector, iteration
        vector = next_vector
        previous_eigenvalue = eigenvalue
    raise RuntimeError("Power method did not converge within max_iterations.")


def successive_power_method(matrix, count, tolerance=1e-7):
    """Apply C - sum(v_j v_j^T C) to find successive nonzero eigenpairs."""
    eigenvalues = []
    eigenvectors = []
    iteration_counts = []
    for _ in range(count):
        deflated = matrix.copy()
        for previous_vector in eigenvectors:
            deflated -= np.outer(previous_vector, previous_vector) @ matrix
        eigenvalue, eigenvector, iterations = power_method(deflated, tolerance)
        eigenvalues.append(eigenvalue)
        eigenvectors.append(eigenvector)
        iteration_counts.append(iterations)
    return np.array(eigenvalues), np.column_stack(eigenvectors), iteration_counts


nonzero_eigenpair_count = matrix_rank_by_elimination(data)
power_tolerance = 1e-9
power_values, power_vectors, iterations = successive_power_method(
    covariance, nonzero_eigenpair_count, tolerance=power_tolerance
)
print_section("Q2.3(b-c) - POWER METHOD AND DEFLATION")
print(f"Nonzero eigenpairs computed: {nonzero_eigenpair_count}")
print(f"Power-method convergence tolerance: {power_tolerance:.8e}")
print_vector("Power-method eigenvalues:", power_values)
print_matrix("Power-method eigenvectors (columns):", power_vectors)
print("Iterations per eigenpair:", iterations)
print("Remaining eigenvalues: two values equal to zero because rank(X) = 4.")


Q2.3(b-c) - POWER METHOD AND DEFLATION
Nonzero eigenpairs computed: 4
Power-method convergence tolerance: 1.00000000e-09
Power-method eigenvalues:
[14.06110983, 5.93222766, 1.00262815, 0.82391310]
Power-method eigenvectors (columns):
[0.16490148, 0.02063009, 0.46601112, -0.68520161]
[0.21066961, -0.03184925, -0.30979546, 0.46451434]
[-0.00251471, 0.20583952, 0.73278608, 0.50396593]
[-0.02683741, -0.35235236, 0.38529136, 0.24497238]
[0.96181178, -0.05428756, 0.00263587, 0.02313980]
[0.05116012, 0.91054423, -0.03779665, 0.01402117]
Iterations per eigenpair: [14, 8, 43, 2]
Remaining eigenvalues: two values equal to zero because rank(X) = 4.


The power method computes the four nonzero eigenpairs. After each eigenvector is found, deflation uses $C - \sum_j v_j v_j^T C$, as specified in the assignment. The remaining two eigenvalues are zero because the covariance matrix has rank 4.

## Q2.3(d): NumPy Comparison


In [47]:
"""Q2.3(d): Compare power-method results with NumPy's eigendecomposition."""

exact_values, exact_vectors = np.linalg.eigh(covariance)
order = np.argsort(exact_values)[::-1]
exact_values = exact_values[order]
exact_vectors = exact_vectors[:, order]
positive_exact_values = exact_values[:nonzero_eigenpair_count]
positive_exact_vectors = exact_vectors[:, :nonzero_eigenpair_count]
value_differences = np.abs(power_values - positive_exact_values)
vector_alignments = []
residuals = []
for index in range(nonzero_eigenpair_count):
    vector_alignments.append(
        abs(power_vectors[:, index] @ positive_exact_vectors[:, index])
    )
    residuals.append(
        np.linalg.norm(
            covariance @ power_vectors[:, index]
            - power_values[index] * power_vectors[:, index]
        )
    )

print_section("Q2.3(d) - NUMPY COMPARISON")
print_vector("NumPy eigenvalues (descending):", exact_values)
print("\nPositive-eigenvalue comparison:")
print("Index | NumPy value | Power value | Absolute difference | Vector alignment | Residual norm")
print("-" * 100)
for index in range(nonzero_eigenpair_count):
    print(
        f"{index + 1:5d} | "
        f"{positive_exact_values[index]:11.8f} | "
        f"{power_values[index]:11.8f} | "
        f"{value_differences[index]:18.8e} | "
        f"{vector_alignments[index]:16.8f} | "
        f"{residuals[index]:12.8e}"
    )
print("\nZero-eigenvalue note: NumPy reports two zero eigenvalues.")
print("Their individual eigenvectors are not unique, so they are compared as a subspace rather than by position.")


Q2.3(d) - NUMPY COMPARISON
NumPy eigenvalues (descending):
[14.06110983, 5.93222766, 1.00262815, 0.82391309, 0.00000000, -0.00000000]

Positive-eigenvalue comparison:
Index | NumPy value | Power value | Absolute difference | Vector alignment | Residual norm
----------------------------------------------------------------------------------------------------
    1 | 14.06110983 | 14.06110983 |     6.87645496e-11 |       1.00000000 | 2.36433611e-05
    2 |  5.93222766 |  5.93222766 |     6.18340934e-11 |       1.00000000 | 2.43564755e-05
    3 |  1.00262815 |  1.00262815 |     1.72554881e-09 |       1.00000000 | 1.85340920e-05
    4 |  0.82391309 |  0.82391310 |     1.73248860e-09 |       1.00000000 | 1.76050466e-05

Zero-eigenvalue note: NumPy reports two zero eigenvalues.
Their individual eigenvectors are not unique, so they are compared as a subspace rather than by position.


The four power-method eigenvalues agree with NumPy's corresponding positive eigenvalues within the requested numerical tolerance. The eigenvector dot products are nearly 1, and the residual norms quantify the remaining approximation error. The two zero-eigenvalue eigenvectors are not compared individually because any orthonormal basis of the null space is valid.

## Q2.3(e): Iteration Comparison


In [48]:
"""Q2.3(e): Compare iterations required to reach 1e-7 eigenvalue accuracy."""


def iterations_to_reference_value(matrix, target, tolerance=1e-7, max_iterations=100000):
    """Count power iterations until the Rayleigh value matches a reference."""
    vector = np.ones(matrix.shape[0], dtype=float)
    vector /= np.linalg.norm(vector)
    for iteration in range(1, max_iterations + 1):
        next_vector = matrix @ vector
        next_vector /= np.linalg.norm(next_vector)
        eigenvalue = float(next_vector @ matrix @ next_vector)
        if abs(eigenvalue - target) < tolerance:
            return iteration, eigenvalue
        vector = next_vector
    raise RuntimeError("Reference accuracy was not reached within max_iterations.")


accuracy = 1e-7
reference_deflated = covariance.copy()
reference_iterations = []
reference_values = []
for index in range(nonzero_eigenpair_count):
    iteration_count, reference_value = iterations_to_reference_value(
        reference_deflated, positive_exact_values[index], accuracy
    )
    reference_iterations.append(iteration_count)
    reference_values.append(reference_value)
    reference_deflated -= (
        np.outer(positive_exact_vectors[:, index], positive_exact_vectors[:, index])
        @ covariance
    )

print_section("Q2.3(e) - ITERATION ACCURACY COMPARISON")
print(f"Target absolute eigenvalue accuracy: {accuracy:.8e}")
print("Index | Iterations | Reference eigenvalue | Absolute error")
print("-" * 62)
for index in range(nonzero_eigenpair_count):
    print(
        f"{index + 1:5d} | "
        f"{reference_iterations[index]:10d} | "
        f"{positive_exact_values[index]:19.8f} | "
        f"{abs(reference_values[index] - positive_exact_values[index]):.8e}"
    )


Q2.3(e) - ITERATION ACCURACY COMPARISON
Target absolute eigenvalue accuracy: 1.00000000e-07
Index | Iterations | Reference eigenvalue | Absolute error
--------------------------------------------------------------
    1 |         10 |         14.06110983 | 6.85175596e-08
    2 |          6 |          5.93222766 | 8.60015525e-09
    3 |         33 |          1.00262815 | 8.78661166e-08
    4 |          1 |          0.82391309 | 3.33066907e-16
